# 🔍 YOLO 모델 성능 평가 & 예측 시각화

## 📁 로컬 폴더 구조
```
d:/DeepLearning/
├── yoloModel/
│   └── battery_yolo11n_best_30epochs.pt   ← 학습된 YOLO 모델
├── dataset/
│   └── battery_subset_binary_train10000/
│       ├── images/
│       │   ├── defect/   ← 불량 이미지 (Damaged_xxx.png, Pollution_xxx.png)
│       │   └── normal/   ← 정상 이미지
│       ├── train.csv
│       └── valid.csv
├── yolo_eval/              ← (자동 생성) 평가 결과 저장 폴더
│   ├── val/               ← val 평가 결과 (confusion_matrix, PR_curve 등)
│   └── predict/           ← predict 시각화 결과 이미지
└── battery_yolo_eval.ipynb ← 이 노트북
```

## ⚠️ YOLO val 평가 vs YOLO predict 차이
| | YOLO val | YOLO predict |
|---|---|---|
| 라벨(.txt) 필요 | ✅ 필수 | ❌ 불필요 |
| 성능 지표 출력 | ✅ mAP, Precision, Recall | ❌ |
| 시각화 | confusion_matrix, PR curve | Bounding Box 그린 이미지 |

**현재 로컬에는 YOLO 형식 라벨(.txt)이 없으므로:**
- **셀 1~4**: predict 시각화 → 즉시 실행 가능 ✅  
- **셀 5~7**: val 평가 → `data.yaml` + 라벨 파일 필요 (Kaggle 학습 환경의 라벨 사용)

In [ ]:
# ==============================================================
# [셀 1] 환경 설정 및 모델 파일 경로 확인
# ==============================================================
# ultralytics: YOLO 계열 모델(YOLOv5~YOLO11)을 쉽게 사용할 수 있는 라이브러리
# YOLO 클래스 하나로 모델 로드, 학습, 평가, 예측이 모두 가능합니다.
# ==============================================================

import os
from pathlib import Path

import cv2
# ==============================================================
# ── matplotlib 한글 폰트 설정 (Windows: 맑은 고딕) ──
# 기본 matplotlib 폰트는 한글을 지원하지 않아 글자가 깨집니다.
# Windows 에는 '맑은 고딕(Malgun Gothic)'이 기본 내장되어 있습니다.
# plt.rcParams['axes.unicode_minus'] = False 는
#   음수 부호(-)가 □□로 깨지는 현상을 방지합니다.
# ==============================================================
import platform
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'   # 맑은 고딕
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

# 음수 부호 깨짐 방지 (반드시 font 설정 후에 지정)
plt.rcParams['axes.unicode_minus'] = False

print(f'✅ 한글 폰트 설정 완료: {plt.rcParams["font.family"]}')

import numpy as np
import pandas as pd
from ultralytics import YOLO

# ── 경로 설정 ──
ROOT         = Path('d:/DeepLearning')                  # 프로젝트 루트
MODEL_PATH   = ROOT / 'yoloModel/battery_yolo11n_best_30epochs.pt'  # 모델 파일
DATASET_ROOT = ROOT / 'dataset/battery_subset_binary_train10000'    # 데이터셋 루트
IMAGE_DIR    = DATASET_ROOT / 'images'                  # 이미지 폴더
VALID_CSV    = DATASET_ROOT / 'valid.csv'               # 검증 CSV
OUTPUT_DIR   = ROOT / 'yolo_eval'                       # 결과 저장 루트
OUTPUT_DIR.mkdir(exist_ok=True)

# ── 모델 파일 확인 ──
print('=' * 60)
print('📁 경로 확인')
print('=' * 60)
print(f'모델 파일 존재: {MODEL_PATH.exists()} → {MODEL_PATH}')
print(f'이미지 폴더 존재: {IMAGE_DIR.exists()} → {IMAGE_DIR}')
print(f'결과 저장 폴더: {OUTPUT_DIR}')

# ── 이미지 수 확인 ──
defect_imgs = list((IMAGE_DIR / 'defect').glob('*.png'))
normal_imgs = list((IMAGE_DIR / 'normal').glob('*.png'))
print(f'\n불량(defect) 이미지: {len(defect_imgs)}장')
print(f'정상(normal) 이미지: {len(normal_imgs)}장')

In [ ]:
# ==============================================================
# [셀 2] YOLO 모델 로드 및 기본 정보 확인
# ==============================================================
# YOLO('경로') 한 줄로 모델을 메모리에 올릴 수 있습니다.
# model.names: 탐지 클래스 이름 딕셔너리 (예: {0: 'Damaged', 1: 'Pollution'})
# model.info(): 모델 레이어 수, 파라미터 수 등 요약 정보 출력
# ==============================================================

print('YOLO 모델 로드 중...')
model = YOLO(str(MODEL_PATH))

print('\n' + '=' * 60)
print('📊 모델 정보')
print('=' * 60)
print(f'모델 파일: {MODEL_PATH.name}')
print(f'탐지 클래스: {model.names}')
print(f'클래스 수: {len(model.names)}')
print(f'\n모델 구조 요약:')
model.info()

In [ ]:
# ==============================================================
# [셀 3] YOLO predict 시각화 - 불량 이미지 샘플 예측
# ==============================================================
# model.predict():
#   - source: 예측할 이미지 경로 (단일, 폴더, URL 모두 가능)
#   - conf: Confidence 임계값 (이 값 이상인 탐지 결과만 출력, 기본 0.25)
#   - save: True면 결과 이미지를 자동 저장
#   - project / name: 저장 폴더 지정
#   - verbose: True면 탐지 과정 상세 출력
#
# 반환값 results: 각 이미지에 대한 탐지 결과 객체 리스트
#   result.boxes.xyxy    → Bounding Box 좌표 (x1, y1, x2, y2)
#   result.boxes.conf    → Confidence Score
#   result.boxes.cls     → 클래스 인덱스
#   result.plot()        → 탐지 결과가 그려진 BGR 이미지 반환
# ==============================================================

CONF_THRESHOLD = 0.25  # Confidence 임계값 (낮을수록 민감하게 탐지)
SAMPLE_COUNT   = 20    # 시각화할 샘플 수

# valid.csv에서 불량(defect) 이미지만 샘플링
valid_df = pd.read_csv(VALID_CSV)
defect_df = valid_df[valid_df['binary_label'] == 'defect'].copy()
defect_df['abs_path'] = defect_df['new_relative_path'].apply(
    lambda x: str(DATASET_ROOT / x.replace('\\', '/'))
)

# 파일 존재 & 크기 > 0 인 것만 필터
defect_df = defect_df[
    defect_df['abs_path'].apply(lambda p: os.path.exists(p) and os.path.getsize(p) > 0)
].copy()

sample_paths = defect_df['abs_path'].sample(min(SAMPLE_COUNT, len(defect_df)), random_state=42).tolist()
print(f'예측할 샘플 수: {len(sample_paths)}장')

# ── YOLO 예측 실행 ──
print('\nYOLO 예측 실행 중...')
results = model.predict(
    source=sample_paths,
    conf=CONF_THRESHOLD,
    save=True,                               # Bounding Box 그린 이미지 저장
    project=str(OUTPUT_DIR / 'predict'),     # 저장 경로
    name='defect_samples',
    exist_ok=True,
    verbose=False
)
print(f'✅ 예측 완료! 결과 저장 위치: {OUTPUT_DIR}/predict/defect_samples')

In [ ]:
# ==============================================================
# [셀 4] 예측 결과 Jupyter 화면에 시각화 (PPT용)
# ==============================================================
# result.plot()는 OpenCV BGR 형식으로 반환되므로
# Matplotlib으로 보려면 cv2.cvtColor로 RGB 변환이 필요합니다.
# ==============================================================

SHOW_COUNT = 12  # Jupyter에 표시할 이미지 수

fig, axes = plt.subplots(3, 4, figsize=(20, 14))
fig.suptitle('YOLO11n 배터리 불량 탐지 예측 결과\n(Damaged / Pollution)', 
             fontsize=18, fontweight='bold', y=1.01)

displayed = 0
for result, ax in zip(results[:SHOW_COUNT], axes.flatten()):
    # result.plot(): 탐지 박스가 그려진 BGR 이미지 반환
    annotated_bgr = result.plot()
    # BGR → RGB 변환 (matplotlib은 RGB 형식 사용)
    annotated_rgb = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)
    
    ax.imshow(annotated_rgb)
    ax.axis('off')
    
    # 탐지 결과 요약 타이틀
    filename = Path(result.path).name[:30]
    n_det = len(result.boxes) if result.boxes else 0
    if n_det > 0:
        confs = result.boxes.conf.cpu().numpy()
        classes = result.boxes.cls.cpu().numpy().astype(int)
        labels = [model.names[c] for c in classes]
        title = f'{filename}\n탐지: {", ".join(labels)} (conf: {confs.mean():.2f})'
        ax.set_title(title, fontsize=8, color='red')
    else:
        ax.set_title(f'{filename}\n탐지 없음', fontsize=8, color='gray')
    displayed += 1

# 남는 subplot 비우기
for ax in axes.flatten()[displayed:]:
    ax.axis('off')

plt.tight_layout()

# PPT용 고해상도 이미지로 저장
save_path = OUTPUT_DIR / 'predict_grid_ppt.png'
plt.savefig(save_path, dpi=200, bbox_inches='tight', facecolor='white')
print(f'✅ PPT용 그리드 이미지 저장: {save_path}')
plt.show()

In [ ]:
# ==============================================================
# [셀 5] data.yaml 생성 (YOLO val 평가를 위한 필수 설정 파일)
# ==============================================================
# YOLO val 평가를 하려면 data.yaml 파일이 필요합니다.
#
# data.yaml 구조:
#   path  : 데이터셋 루트 경로
#   train : 학습 이미지 경로 (상대경로)
#   val   : 검증 이미지 경로 (상대경로)
#   nc    : Number of Classes (클래스 수)
#   names : 클래스 이름 리스트
#
# ⚠️ 중요: YOLO val은 이미지와 같은 위치(또는 ../labels/ 경로)에
#           YOLO 형식의 라벨 .txt 파일이 반드시 있어야 합니다!
#
# YOLO 라벨 .txt 형식 (한 줄 = 한 객체):
#   [클래스인덱스] [x_center] [y_center] [width] [height]
#   (모두 0~1 범위의 정규화된 값)
#   예: 0 0.512 0.483 0.142 0.238
#
# 현재 로컬에는 이 라벨 .txt가 없으므로,
# Kaggle에서 학습 시 사용했던 라벨 파일을 다운로드해야 합니다.
# ==============================================================

import yaml

data_yaml_content = {
    'path': str(DATASET_ROOT).replace('\\', '/'),  # 데이터셋 루트 (절대 경로)
    'train': 'images/defect',   # 학습 이미지 경로 (val 평가에선 사용 안 함)
    'val': 'images/defect',     # 검증 이미지 경로 (defect 폴더 사용)
    'nc': 2,                    # 클래스 수: Damaged, Pollution
    'names': ['Damaged', 'Pollution']  # 클래스 이름
}

yaml_path = OUTPUT_DIR / 'data.yaml'
with open(yaml_path, 'w', encoding='utf-8') as f:
    yaml.dump(data_yaml_content, f, allow_unicode=True, default_flow_style=False, sort_keys=False)

print(f'✅ data.yaml 생성 완료: {yaml_path}')
print('\n📄 data.yaml 내용:')
print('-' * 40)
with open(yaml_path, 'r') as f:
    print(f.read())
print('-' * 40)
print('\n⚠️  val 평가를 실행하려면 각 이미지와 동일한 위치에')
print('    YOLO 형식 라벨 .txt 파일이 필요합니다.')
print('    (Kaggle 학습 데이터의 라벨 파일을 활용하세요)')

In [ ]:
# ==============================================================
# [셀 6] YOLO val 평가 실행 (라벨 파일이 있는 경우)
# ==============================================================
# model.val():
#   - data: data.yaml 경로
#   - split: 평가할 데이터 분할 ('val', 'test', 'train')
#   - conf: Confidence 임계값
#   - iou: IoU 임계값 (NMS 적용 기준)
#   - save_json: COCO 형식 JSON으로 저장
#   - plots: True면 confusion_matrix, PR_curve 등 그래프 자동 저장
#
# 반환값 metrics:
#   metrics.box.map50    → mAP@0.5
#   metrics.box.map      → mAP@0.5:0.95
#   metrics.box.mp       → Mean Precision
#   metrics.box.mr       → Mean Recall
# ==============================================================

import os

LABELS_EXIST = False  # 라벨 파일이 없으면 False로 유지

# 라벨 파일 존재 여부 자동 확인
label_check_path = IMAGE_DIR / 'defect'
txt_files = list(label_check_path.glob('*.txt'))
if len(txt_files) > 0:
    LABELS_EXIST = True
    print(f'✅ 라벨 파일 발견: {len(txt_files)}개 → val 평가 실행 가능')
else:
    print('⚠️  라벨 파일(.txt)이 없습니다. val 평가를 건너뜁니다.')
    print('   Kaggle에서 학습 시 사용한 labels/ 폴더를 로컬에 복사해 주세요.')

if LABELS_EXIST:
    print('\nYOLO val 평가 실행 중...')
    metrics = model.val(
        data=str(yaml_path),
        split='val',
        conf=0.25,
        iou=0.5,
        plots=True,                           # confusion_matrix, PR_curve 등 저장
        project=str(OUTPUT_DIR / 'val'),      # 저장 경로
        name='battery_yolo11n',
        exist_ok=True,
        verbose=True
    )

    print('\n' + '=' * 60)
    print('📊 YOLO val 평가 결과')
    print('=' * 60)
    print(f'Precision  (mP)    : {metrics.box.mp:.4f}')
    print(f'Recall     (mR)    : {metrics.box.mr:.4f}')
    print(f'mAP@0.50           : {metrics.box.map50:.4f}')
    print(f'mAP@0.50:0.95      : {metrics.box.map:.4f}')
    print('=' * 60)
    print(f'\n📁 결과 이미지 저장 위치: {OUTPUT_DIR}/val/battery_yolo11n/')
    print('   - confusion_matrix.png')
    print('   - PR_curve.png')
    print('   - F1_curve.png')
    print('   - P_curve.png')
    print('   - R_curve.png')

In [ ]:
# ==============================================================
# [셀 7] 탐지 통계 분석 및 PPT용 요약 차트 생성
# ==============================================================
# 예측 결과(results)에서 탐지된 클래스별 수를 분석하고
# 막대 그래프로 시각화합니다.
# ==============================================================

from collections import Counter

# ── 탐지 통계 집계 ──
stats = {
    'total_images': len(results),
    'detected_images': 0,
    'not_detected': 0,
    'class_counts': Counter(),
    'conf_scores': []
}

for result in results:
    if result.boxes is None or len(result.boxes) == 0:
        stats['not_detected'] += 1
        continue
    
    stats['detected_images'] += 1
    classes = result.boxes.cls.cpu().numpy().astype(int)
    confs = result.boxes.conf.cpu().numpy()
    
    for cls_id, conf in zip(classes, confs):
        class_name = model.names[cls_id]
        stats['class_counts'][class_name] += 1
        stats['conf_scores'].append(conf)

print('=' * 50)
print('📊 탐지 결과 통계 요약')
print('=' * 50)
print(f'총 예측 이미지      : {stats["total_images"]}장')
print(f'탐지 성공 이미지    : {stats["detected_images"]}장')
print(f'탐지 없음 이미지    : {stats["not_detected"]}장')
print(f'클래스별 탐지 수    : {dict(stats["class_counts"])}')
if stats['conf_scores']:
    print(f'평균 Confidence     : {np.mean(stats["conf_scores"]):.4f}')
    print(f'최소 Confidence     : {np.min(stats["conf_scores"]):.4f}')
    print(f'최대 Confidence     : {np.max(stats["conf_scores"]):.4f}')

# ── PPT용 요약 차트 ──
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('YOLO11n 배터리 불량 탐지 결과 통계', fontsize=16, fontweight='bold')

# 왼쪽: 탐지 여부 파이 차트
labels_pie = ['탐지 성공', '탐지 없음']
values_pie = [stats['detected_images'], stats['not_detected']]
colors_pie = ['#ef4444', '#22c55e']
axes[0].pie(values_pie, labels=labels_pie, colors=colors_pie, autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 13})
axes[0].set_title(f'탐지 여부 (총 {stats["total_images"]}장)', fontsize=13)

# 오른쪽: 클래스별 탐지 수 막대 그래프
if stats['class_counts']:
    class_names = list(stats['class_counts'].keys())
    class_vals  = list(stats['class_counts'].values())
    bar_colors  = ['#3b82f6', '#f97316']
    bars = axes[1].bar(class_names, class_vals, color=bar_colors[:len(class_names)], 
                       edgecolor='white', width=0.5)
    for bar, val in zip(bars, class_vals):
        axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3,
                     str(val), ha='center', va='bottom', fontsize=14, fontweight='bold')
    axes[1].set_title('클래스별 탐지 수', fontsize=13)
    axes[1].set_ylabel('탐지 횟수')
    axes[1].set_facecolor('#f8fafc')

plt.tight_layout()

# 저장
chart_path = OUTPUT_DIR / 'detection_stats_ppt.png'
plt.savefig(chart_path, dpi=200, bbox_inches='tight', facecolor='white')
print(f'\n✅ PPT용 통계 차트 저장: {chart_path}')
plt.show()

print('\n' + '=' * 50)
print('📁 최종 결과 파일 목록')
print('=' * 50)
for f in OUTPUT_DIR.rglob('*'):
    if f.is_file():
        print(f'  {f.relative_to(OUTPUT_DIR)}')

---
## 🏷️ YOLO 라벨(.txt) 자동 생성 → val 평가

### ⚠️ 왜 라벨이 없을까?
YOLO val 평가는 각 이미지에 대응하는 **YOLO 형식 `.txt` 라벨 파일**이 필요합니다.
- 원본 JSON 어노테이션은 `D:\DL_dataset\...` 경로에 있었으나 현재 없는 상태입니다.

### 💡 대안: CSV로 '이미지 전체 박스' 라벨 생성
CSV의 `defect_names` 컬럼에는 **각 이미지에 어떤 불량이 있는지** 정보가 있습니다.
정확한 바운딩 박스 좌표는 없지만, **이미지 전체(x=0.5, y=0.5, w=1.0, h=1.0)를**
**하나의 탐지 영역으로 간주**하는 라벨을 만들 수 있습니다.

```
YOLO 라벨 형식 (.txt, 한 줄 = 객체 하나):
  [class_id] [x_center] [y_center] [width] [height]  ← 모두 0~1 정규화 값

  class 0 = Damaged
  class 1 = Pollution

  예) Damaged 이미지 → 0 0.5 0.5 1.0 1.0
      Pollution 이미지 → 1 0.5 0.5 1.0 1.0
      Damaged+Pollution → 두 줄 모두 기록
```

> **주의**: 이 라벨은 '이미지 전체'를 박스로 보기 때문에
> mAP 수치가 실제 정밀 라벨 기반 평가보다 높게 나올 수 있습니다.
> **클래스별 Precision / Recall 경향** 파악 용도로 해석해 주세요.


In [ ]:
# ==============================================================
# [셀 8] CSV → YOLO 라벨(.txt) 자동 생성
# ==============================================================
# YOLO 라벨 파일 위치 규칙:
#   이미지 경로: images/defect/Damaged_xxx.png
#   라벨 경로:   labels/defect/Damaged_xxx.txt  ← 'images'를 'labels'로 교체
#
# 클래스 매핑:
#   0 = Damaged
#   1 = Pollution
#   (normal은 라벨 없음 = 빈 파일)
# ==============================================================

import os
import pandas as pd
from pathlib import Path

ROOT         = Path('d:/DeepLearning')
DATASET_ROOT = ROOT / 'dataset/battery_subset_binary_train10000'
IMAGE_DIR    = DATASET_ROOT / 'images'
LABEL_DIR    = DATASET_ROOT / 'labels'   # 라벨 저장 위치 (자동 생성)

# 클래스 이름 → 인덱스 매핑
CLASS_MAP = {'Damaged': 0, 'Pollution': 1}

# 전체 CSV 로드
valid_df = pd.read_csv(DATASET_ROOT / 'valid.csv')
train_df = pd.read_csv(DATASET_ROOT / 'train.csv')
all_df = pd.concat([train_df, valid_df], ignore_index=True)

print(f'전체 데이터: {len(all_df)}행')
print(f'valid 데이터: {len(valid_df)}행')

created = 0
skipped = 0

for _, row in all_df.iterrows():
    rel_path = str(row['new_relative_path']).replace('\\', '/')
    abs_img  = DATASET_ROOT / rel_path

    # 파일 존재 + 크기 확인
    if not abs_img.exists() or abs_img.stat().st_size == 0:
        skipped += 1
        continue

    # 라벨 저장 경로: images/ → labels/ 교체
    rel_label = rel_path.replace('images/', 'labels/').replace('.png', '.txt').replace('.jpg', '.txt')
    abs_label = DATASET_ROOT / rel_label
    abs_label.parent.mkdir(parents=True, exist_ok=True)

    binary_label = str(row['binary_label'])
    defect_names_raw = str(row.get('defect_names', ''))

    lines = []

    if binary_label == 'normal':
        # 정상 이미지 → 빈 라벨 파일 (객체 없음)
        lines = []
    else:
        # 불량 이미지 → defect_names에서 클래스 파싱
        # defect_names 예시: "Pollution,Damaged,Pollution" or "Damaged"
        present_classes = set()
        for name in defect_names_raw.split(','):
            name = name.strip()
            if name in CLASS_MAP:
                present_classes.add(name)

        # 탐지된 각 클래스마다 이미지 전체를 박스로 기록
        # YOLO 형식: class_id x_center y_center width height (0~1 정규화)
        for cls_name in sorted(present_classes):  # 재현성을 위해 정렬
            cls_id = CLASS_MAP[cls_name]
            lines.append(f'{cls_id} 0.5 0.5 1.0 1.0')

    with open(abs_label, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines))

    created += 1

print(f'\n✅ 라벨 생성 완료: {created}개')
print(f'   건너뜀 (파일 없음): {skipped}개')
print(f'\n라벨 저장 위치:')
print(f'  {LABEL_DIR / "defect"}')
print(f'  {LABEL_DIR / "normal"}')

# 샘플 확인
sample_labels = list((LABEL_DIR / 'defect').glob('*.txt'))[:3]
print('\n📄 샘플 라벨 내용:')
for lbl in sample_labels:
    content = lbl.read_text().strip()
    print(f'  {lbl.name}: [{content if content else "(빈 파일 - 정상)"}]')


In [ ]:
# ==============================================================
# [셀 9] data.yaml 생성 + YOLO val 평가 실행
# ==============================================================
# val() 실행 시 YOLO는 이미지 경로에서 자동으로
# 'images' → 'labels' 로 바꿔 라벨 파일을 찾습니다.
# 예) images/defect/Damaged_xxx.png
#  →  labels/defect/Damaged_xxx.txt
# ==============================================================

import yaml
from ultralytics import YOLO
from pathlib import Path

ROOT         = Path('d:/DeepLearning')
MODEL_PATH   = ROOT / 'yoloModel/battery_yolo11n_best_30epochs.pt'
DATASET_ROOT = ROOT / 'dataset/battery_subset_binary_train10000'
OUTPUT_DIR   = ROOT / 'yolo_eval'
OUTPUT_DIR.mkdir(exist_ok=True)

# data.yaml 생성
# YOLO val의 val 경로는 labels/ 폴더가 같은 위치에 있어야 합니다.
data_yaml = {
    'path': str(DATASET_ROOT).replace('\\', '/'),
    'train': 'images/defect',   # 학습 경로 (val에서는 사용 안 함)
    'val':   'images/defect',   # 검증 이미지 경로 (불량 이미지만 평가)
    'nc': 2,
    'names': ['Damaged', 'Pollution']
}

yaml_path = OUTPUT_DIR / 'data.yaml'
with open(yaml_path, 'w', encoding='utf-8') as f:
    yaml.dump(data_yaml, f, allow_unicode=True, default_flow_style=False, sort_keys=False)

print('✅ data.yaml 생성 완료')
print()

# 모델 로드
model = YOLO(str(MODEL_PATH))

# val 평가 실행
print('YOLO val 평가 실행 중...')
metrics = model.val(
    data=str(yaml_path),
    split='val',
    conf=0.25,
    iou=0.5,
    plots=True,                          # confusion_matrix, PR_curve 등 자동 저장
    project=str(OUTPUT_DIR / 'val'),     # 결과 저장 폴더
    name='battery_yolo11n',
    exist_ok=True,
    verbose=True,
)

print('\n' + '=' * 60)
print('📊 YOLO val 평가 결과')
print('=' * 60)
print(f'  Precision  (mP)    : {metrics.box.mp:.4f}')
print(f'  Recall     (mR)    : {metrics.box.mr:.4f}')
print(f'  mAP@0.50           : {metrics.box.map50:.4f}')
print(f'  mAP@0.50:0.95      : {metrics.box.map:.4f}')
print('=' * 60)
print(f'\n📁 결과 이미지 저장 위치: {OUTPUT_DIR}/val/battery_yolo11n/')
print('   - confusion_matrix.png  ← PPT 활용')
print('   - PR_curve.png          ← PPT 활용')
print('   - F1_curve.png          ← PPT 활용')
print('   - P_curve.png')
print('   - R_curve.png')


In [ ]:
# ==============================================================
# [셀 10] val 결과 이미지 Jupyter 화면에 표시 (PPT용)
# ==============================================================

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import platform
from pathlib import Path

# 한글 폰트 재적용
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

OUTPUT_DIR = Path('d:/DeepLearning/yolo_eval')
val_dir    = OUTPUT_DIR / 'val' / 'battery_yolo11n'

# 표시할 결과 이미지 목록
target_files = [
    ('confusion_matrix.png', 'Confusion Matrix'),
    ('PR_curve.png',          'PR Curve (Precision-Recall)'),
    ('F1_curve.png',          'F1 Curve'),
    ('P_curve.png',           'Precision Curve'),
    ('R_curve.png',           'Recall Curve'),
]

available = [(fname, title) for fname, title in target_files
             if (val_dir / fname).exists()]

if not available:
    print(f'⚠️  결과 이미지가 없습니다. 셀 9 (val 평가)를 먼저 실행해주세요.')
    print(f'   확인 경로: {val_dir}')
else:
    n = len(available)
    cols = min(n, 3)
    rows = (n + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(8 * cols, 6 * rows))
    if n == 1:
        axes = [[axes]]
    elif rows == 1:
        axes = [axes]

    for idx, (fname, title) in enumerate(available):
        r, c = divmod(idx, cols)
        img = mpimg.imread(str(val_dir / fname))
        axes[r][c].imshow(img)
        axes[r][c].set_title(title, fontsize=14, fontweight='bold')
        axes[r][c].axis('off')

    # 남은 subplot 숨기기
    for idx in range(len(available), rows * cols):
        r, c = divmod(idx, cols)
        axes[r][c].axis('off')

    plt.suptitle('YOLO11n 배터리 불량 탐지 - val 평가 결과', fontsize=18, fontweight='bold')
    plt.tight_layout()

    # PPT용 저장
    ppt_path = OUTPUT_DIR / 'val_results_ppt.png'
    plt.savefig(ppt_path, dpi=200, bbox_inches='tight', facecolor='white')
    print(f'✅ PPT용 결과 이미지 저장: {ppt_path}')
    plt.show()
